In [ ]:
!pip install pdfplumber pytesseract pillow camelot-py[cv] tabula-py
!pip install sentence-transformers faiss-cpu
!pip install streamlit langchain
!apt-get install -y ghostscript tesseract-ocr


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest v

In [ ]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Folders created successfully.")


Folders created successfully.


In [ ]:
from google.colab import files

uploaded = files.upload()  # upload your sample PDFs


Saving qatar_test_doc.pdf to qatar_test_doc.pdf


In [ ]:
import pdfplumber
import pytesseract
from PIL import Image
import camelot
import json
import os

pdf_path = "/content/qatar_test_doc.pdf"

output = []
page_num = 1

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_data = {"page": page_num}

        # ---------- 1. Extract Raw Text ----------
        text = page.extract_text() or ""
        page_data["text"] = text

        # ---------- 2. Extract Images + OCR ----------
        ocr_text = ""
        images = page.images

        image_texts = []
        for i, im in enumerate(images):
            # crop image area
            try:
                bbox = (im["x0"], im["top"], im["x1"], im["bottom"])
                cropped = page.crop(bbox).to_image(resolution=300)
                img_path = f"outputs/page_{page_num}_img_{i}.png"
                cropped.save(img_path)

                # OCR
                text_from_img = pytesseract.image_to_string(Image.open(img_path))
                image_texts.append(text_from_img)
            except:
                pass

        page_data["images_ocr"] = image_texts

        # ---------- 3. Extract Tables ----------
        try:
            tables = camelot.read_pdf(pdf_path, pages=str(page_num))
            table_texts = [table.df.to_json() for table in tables]
            page_data["tables"] = table_texts
        except:
            page_data["tables"] = []

        output.append(page_data)
        page_num += 1

# ---------- Save as JSONL ----------
with open("outputs/parsed_pages.jsonl", "w", encoding="utf-8") as f:
    for entry in output:
        f.write(json.dumps(entry, ensure_ascii=False))
        f.write("\n")

print("PDF extraction finished. Saved to outputs/parsed_pages.jsonl")


/usr/local/lib/python3.12/dist-packages/camelot/utils.py:1217: UserWarning:   (546.24, 547.5) does not lie in column range (67.38, 544.68)
  warnings.warn(


PDF extraction finished. Saved to outputs/parsed_pages.jsonl


In [ ]:
import json

with open("outputs/parsed_pages.jsonl") as f:
    for i in range(3):  # print first 3 pages
        print(json.loads(next(f)))


{'page': 1, 'text': 'IMF Country Report No. 25/47\nQATAR\n2024 ARTICLE IV CONSULTATION—PRESS RELEASE;\nFebruary 2025 STAFF REPORT; AND STATEMENT BY THE EXECUTIVE\nDIRECTOR FOR QATAR\nUnder Article IV of the IMF’s Articles of Agreement, the IMF holds bilateral discussions\nwith members, usually every year. In the context of the 2024 Article IV consultation with\nQatar, the following documents have been released and are included in this package:\n• A Press Release summarizing the views of the Executive Board as expressed during its\nJanuary 27, 2025, consideration of the staff report that concluded the Article IV\nconsultation with Qatar.\n• The Staff Report prepared by a staff team of the IMF for the Executive Board’s\nconsideration on January 27, 2025, following discussions that ended on\nNovember 21, 2024, with the officials of Qatar on economic developments and\npolicies. Based on information available at the time of these discussions, the staff\nreport was completed on January 8, 20

In [ ]:
# STEP 3: Chunking, embeddings, FAISS index
import json, os, math, pickle
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from pathlib import Path

# paths
PARSED_PATH = "outputs/parsed_pages.jsonl"
INDEX_DIR = "outputs/faiss_index"
os.makedirs(INDEX_DIR, exist_ok=True)

# 1) Read parsed pages
pages = []
with open(PARSED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        pages.append(json.loads(line))

# 2) Compose text blocks per page (text + OCR images + tables)
docs = []
for p in pages:
    page_num = p.get("page", None)
    parts = []
    if p.get("text"):
        parts.append(p["text"])
    if p.get("images_ocr"):
        # join non-empty OCR results
        img_txt = "\n".join([t for t in p["images_ocr"] if t and t.strip()])
        if img_txt.strip():
            parts.append("IMAGES_OCR: " + img_txt)
    if p.get("tables"):
        # tables are json strings of dataframes; include a short marker
        if len(p["tables"])>0:
            parts.append("TABLES_PRESENT")
    full = "\n\n".join(parts).strip()
    if not full:
        full = ""  # keep placeholders if you want
    docs.append({"page": page_num, "text": full})

# 3) Chunking function - simple char-based with overlap
def chunk_text(text, chunk_size=800, overlap=200):
    if not text:
        return []
    chunks = []
    start = 0
    L = len(text)
    while start < L:
        end = min(start + chunk_size, L)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == L:
            break
        start = max(0, end - overlap)
    return chunks

# 4) Build corpus of chunks with metadata
corpus = []  # list of dicts: {id, text, page}
chunk_id = 0
for d in docs:
    page = d["page"]
    text = d["text"]
    chunks = chunk_text(text, chunk_size=1000, overlap=200)
    if not chunks:
        # create a tiny placeholder chunk so pages with only tables/images are searchable
        chunks = [f"[blank page {page} or only table/image content]"]
    for c in chunks:
        corpus.append({"id": chunk_id, "page": page, "text": c})
        chunk_id += 1

print(f"Total chunks: {len(corpus)}")

# 5) Create embeddings with sentence-transformers
model_name = "all-MiniLM-L6-v2"
print("Loading embedding model:", model_name)
embed_model = SentenceTransformer(model_name)

texts = [c["text"] for c in corpus]
# batch encode
batch_size = 64
embs = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    e = embed_model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embs.append(e)
emb_matrix = np.vstack(embs).astype("float32")
print("Embeddings shape:", emb_matrix.shape)

# 6) Build FAISS index (cosine via inner product on normalized vectors)
d = emb_matrix.shape[1]
index = faiss.IndexFlatIP(d)  # inner product index
# normalize vectors to use inner product as cosine similarity
faiss.normalize_L2(emb_matrix)
index.add(emb_matrix)
print("FAISS index built. n_items =", index.ntotal)

# 7) Save index and metadata
faiss.write_index(index, os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "corpus_metadata.pkl"), "wb") as f:
    pickle.dump(corpus, f)
print("Saved FAISS index and metadata to", INDEX_DIR)

# 8) Provide a helper retrieval function (local)
def retrieve(query, top_k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx == -1:
            continue
        meta = corpus[idx]
        results.append({"score": float(score), "page": meta["page"], "text": meta["text"]})
    return results

# 9) Quick test query (change the query to something relevant)
test_query = "What are the main points about Qatar's economic policy?"
print("Test query:", test_query)
res = retrieve(test_query, top_k=5)
for i, r in enumerate(res):
    print(f"\nResult {i+1} — score: {r['score']:.4f} — page: {r['page']}")
    snippet = r['text'][:800].replace("\n", " ")
    print(snippet)

# Save a small JSON summary for quick inspection
with open(os.path.join(INDEX_DIR, "index_summary.json"), "w", encoding="utf-8") as f:
    json.dump({"n_chunks": len(corpus), "model": model_name, "index_items": index.ntotal}, f, ensure_ascii=False, indent=2)

print("STEP 3 finished — embeddings + FAISS ready.")


Total chunks: 274
Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (274, 384)
FAISS index built. n_items = 274
Saved FAISS index and metadata to outputs/faiss_index
Test query: What are the main points about Qatar's economic policy?

Result 1 — score: 0.6942 — page: 41
QATAR Table 3a. Qatar: Summary of Central Government Finance, 2020–29 (Billions of Qatari Riyals unless otherwise noted) Projections 2020 2021 2022 2023 2024 2025 2026 2027 2028 2029 Revenue 171.2 193.7 297.8 254.4 213.5 238.2 266.4 275.8 281.6 293.2 Oil 32.3 55.2 57.5 39.0 33.5 30.3 29.6 29.6 30.2 30.9 LNG 35.5 58.4 117.3 88.3 80.2 111.6 127.8 133.0 133.0 140.4 Investment income from public enterprises 1/ 65.5 42.7 73.7 75.2 55.0 51.0 55.3 56.5 57.1 58.3 Corporate tax revenue 1/ 23.7 21.1 27.9 34.6 28.4 28.5 36.4 39.1 43.6 45.6 Other revenue 1/ 14.2 16.2 21.4 17.3 16.3 16.8 17.2 17.6 17.8 17.9 Expenditure 182.4 192.1 208.7 211.4 210.8 217.4 224.8 230.8 238.1 245.7 Expense 115.9 119.8 133.4 136.0 139.9 145.1 150.9 155.3 160.7 166.4 Compensation of employees 57.9 58.7 6

In [ ]:
from groq import Groq
import os

def generate_openai_answer(prompt, openai_api_key=None, model="llama-3.3-70b-versatile", temperature=0.0, max_tokens=400):

    if openai_api_key is None:
        openai_api_key = os.environ.get("GROQ_API_KEY")

    if not openai_api_key:
        raise RuntimeError("GROQ API key not found. Set it using os.environ['GROQ_API_KEY'].")

    client = Groq(api_key=openai_api_key)

    messages = [
        {"role": "system", "content": "You answer using the provided PDF context and cite page numbers."},
        {"role": "user", "content": prompt}
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    answer = resp.choices[0].message["content"].strip()
    return answer, resp


In [ ]:
!pip install groq pyngrok streamlit sentence-transformers faiss-cpu --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

print("Groq key set:", bool(os.environ.get("GROQ_API_KEY")))


Groq key set: True


In [ ]:
!pip install PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 51.9 MB/s eta 0:00:00


In [ ]:
!pip install easyocr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 12.6 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import faiss
import fitz  # PyMuPDF
import numpy as np
import os
import pickle
import tempfile
import camelot
from groq import Groq
from sentence_transformers import SentenceTransformer
import easyocr

# -----------------------------------------
# CONFIG
# -----------------------------------------
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
MODEL_NAME = "llama-3.3-70b-versatile"

# OCR Reader
@st.cache_resource
def load_ocr():
    return easyocr.Reader(["en"])

# Embedding Model
@st.cache_resource
def load_embedder():
    return SentenceTransformer(EMBED_MODEL_NAME)

embed_model = load_embedder()
ocr_reader = load_ocr()


# -----------------------------------------
# PDF INGESTION PIPELINE
# -----------------------------------------
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc):
        text = page.get_text()
        pages.append({"page": page_num + 1, "text": text})
    return pages


def extract_tables(pdf_path):
    try:
        tables = camelot.read_pdf(pdf_path, pages="all")
        extracted = []
        for t in tables:
            extracted.append(str(t.df))
        return extracted
    except:
        return []


def extract_images_and_ocr(pdf_path):
    doc = fitz.open(pdf_path)
    ocr_text = []

    for page_index, page in enumerate(doc):
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            pix = fitz.Pixmap(doc, xref)

            if pix.n < 5:  # RGB
                img_bytes = pix.tobytes("png")
            else:  # CMYK -> convert to RGB
                pix = fitz.Pixmap(fitz.csRGB, pix)
                img_bytes = pix.tobytes("png")

            with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tf:
                tf.write(img_bytes)
                tf.flush()

                # OCR
                text = ocr_reader.readtext(tf.name, detail=0)
                text = " ".join(text)

                ocr_text.append({
                    "page": page_index + 1,
                    "text": text
                })

    return ocr_text


# -----------------------------------------
# CHUNKING
# -----------------------------------------
def chunk_text(text, chunk_size=500):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks


# -----------------------------------------
# INDEX BUILDING
# -----------------------------------------
def build_faiss_index(all_chunks):
    embeddings = embed_model.encode(all_chunks, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index, embeddings


# -----------------------------------------
# RETRIEVAL
# -----------------------------------------
def retrieve_top_k(index, query, k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, ids = index.search(q_emb, k)

    return ids[0]


# -----------------------------------------
# LLM (GROQ)
# -----------------------------------------
def call_groq(prompt):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "❌ Missing GROQ_API_KEY", None

    client = Groq(api_key=api_key)
    try:
        res = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Answer using only provided context"},
                {"role": "user", "content": prompt},
            ],
            temperature=0
        )
        return res.choices[0].message.content, res
    except Exception as e:
        return f"❌ Error: {e}", None


# -----------------------------------------
# STREAMLIT UI
# -----------------------------------------
st.title("📄 Multi-Modal RAG QA System (Groq + OCR + Tables)")

uploaded_pdf = st.file_uploader("Upload a PDF document", type=["pdf"])

if uploaded_pdf:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tf:
        tf.write(uploaded_pdf.read())
        pdf_path = tf.name

    st.success("PDF uploaded successfully!")

    # Extract
    with st.spinner("Extracting text..."):
        pages = extract_text(pdf_path)

    with st.spinner("Extracting tables..."):
        tables = extract_tables(pdf_path)

    with st.spinner("Extracting images + OCR..."):
        ocr_blocks = extract_images_and_ocr(pdf_path)

    # Build chunk list
    all_chunks = []
    page_map = []

    for p in pages:
        chunks = chunk_text(p["text"])
        for c in chunks:
            all_chunks.append(c)
            page_map.append(p["page"])

    for t in tables:
        all_chunks.append(t)
        page_map.append(-1)

    for o in ocr_blocks:
        chunks = chunk_text(o["text"])
        for c in chunks:
            all_chunks.append(c)
            page_map.append(o["page"])

    st.success(f"Indexed {len(all_chunks)} chunks of data!")

    with st.spinner("Building FAISS index..."):
        index, embeddings = build_faiss_index(all_chunks)

    st.success("Index ready! You can now ask questions.")

    question = st.text_input("Ask a question about the document:")

    if st.button("Search"):
        top_ids = retrieve_top_k(index, question)
        retrieved_context = ""

        for idx in top_ids:
            retrieved_context += f"[Chunk from page {page_map[idx]}]\n{all_chunks[idx]}\n\n"

        prompt = f"""
Use ONLY the following context to answer the question.
Cite pages like [page 5].

Context:
{retrieved_context}

Question: {question}
"""

        answer, raw = call_groq(prompt)

        st.subheader("Answer:")
        st.write(answer)

        st.subheader("Retrieved Sources:")
        st.write(retrieved_context)


Writing app.py


In [ ]:
import streamlit as st
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
from PyPDF2 import PdfReader

# ----------------------------
# CONFIG
# ----------------------------
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
MODEL_NAME = "llama-3.3-70b-versatile"   # Groq Model

# ----------------------------
# LOAD EMBEDDING MODEL
# ----------------------------
@st.cache_resource
def load_model():
    return SentenceTransformer(EMBED_MODEL_NAME)

embed_model = load_model()

# ----------------------------
# STREAMLIT UI TITLE
# ----------------------------
st.title("📄 Multi-Modal RAG QA System (Groq Powered)")
st.write("Upload PDF → Ask Question → Get Answer with Citations")

# ----------------------------
# PDF UPLOAD SECTION
# ----------------------------
uploaded_pdf = st.file_uploader("📤 Upload a PDF", type=["pdf"])

index = None
corpus = None

if uploaded_pdf:
    st.write("⏳ Processing PDF...")

    pdf_path = "uploaded.pdf"
    with open(pdf_path, "wb") as f:
        f.write(uploaded_pdf.read())

    reader = PdfReader(pdf_path)

    documents = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if not text:
            continue
        documents.append({"page": i + 1, "text": text})

    st.success("PDF Loaded Successfully!")

    # Build embeddings
    texts = [d["text"] for d in documents]
    embeddings = embed_model.encode(texts, convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(embeddings)

    # Create FAISS index
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    corpus = documents

    st.info("You can now ask questions about the uploaded PDF.")

# ----------------------------
# RETRIEVE CHUNKS
# ----------------------------
def retrieve_chunks(query, top_k=5):
    if index is None:
        return []

    q_emb = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)

    D, I = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(D[0], I[0]):
        meta = corpus[idx]
        results.append({
            "score": float(score),
            "page": meta["page"],
            "text": meta["text"],
        })
    return results

# ----------------------------
# PROMPT BUILDER
# ----------------------------
PROMPT_INSTRUCTIONS = """
You are a helpful assistant who answers ONLY using the provided document excerpts.
Cite pages like [page 3]. If the answer is not found, say "Not found in the document".
"""

def build_prompt(question, chunks):
    context = ""
    for ch in chunks:
        snippet = ch["text"].replace("\n", " ")
        snippet = snippet[:800] + "..." if len(snippet) > 800 else snippet
        context += f"(page {ch['page']}): {snippet}\n\n"

    return f"""
{PROMPT_INSTRUCTIONS}

Context:
{context}

Question: {question}

Answer clearly with citations like [page X].
"""

# ----------------------------
# GROQ LLM CALL
# ----------------------------
def call_groq(prompt):
    api_key = os.environ.get("GROQ_API_KEY")

    if not api_key:
        return "❌ GROQ_API_KEY is missing. Set it using os.environ.", None

    client = Groq(api_key=api_key)

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "Answer using ONLY the provided context."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        answer = response.choices[0].message.content
        return answer, response

    except Exception as e:
        return f"❌ Error: {str(e)}", None


# ----------------------------
# QUESTION UI
# ----------------------------
question = st.text_input("Ask a question about the document:")
go = st.button("Search")

if go:
    if index is None:
        st.warning("Please upload a PDF first.")
    elif not question.strip():
        st.warning("Please enter a question.")
    else:
        st.write("🔍 Retrieving relevant document chunks...")
        chunks = retrieve_chunks(question)

        st.write("📝 Building prompt...")
        prompt = build_prompt(question, chunks)

        st.write("🤖 Generating answer using Groq...")
        answer, raw = call_groq(prompt)

        st.subheader("Answer:")
        st.write(answer)

        st.subheader("Sources:")
        for ch in chunks:
            st.write(f"**Page {ch['page']}** — {ch['text'][:300].replace('\n', ' ')}...")


2025-12-03 09:52:32.633 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:32.910 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-03 09:52:32.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:32.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:32.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:33.419 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:33.421 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-03 09:52:33.422 Thread 'Thread-5': missing 

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.2 MB/s eta 0:00:00


In [ ]:
!pkill -f streamlit || true
!nohup streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &


^C
nohup: appending output to 'nohup.out'


In [ ]:
#####

In [ ]:
#####

In [ ]:
from pyngrok import ngrok

# Ensure your ngrok auth token is set here or in a preceding cell
# If you have a different token, replace it here.
ngrok.set_auth_token("")

ngrok.kill() # Kills any ngrok processes running in this environment
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://44b4e4b51433.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("")
